# 🎙️ KHM-TtS: Khmer Neural Text-to-Speech Training Pipeline
### ការបង្វឹកគំរូសំយោគសំឡេងខ្មែរ (FastSpeechLite + HiFiGAN) លើ Google Colab

សៀវភៅកត់ត្រានេះ (Jupyter Notebook) ត្រូវបានរៀបចំឡើងជាពិសេសដើម្បី Train គំរូសំឡេង KHM-TtS ជាមួយទិន្នន័យ OpenSLR 42 (`km_kh_male`, 2,906 utterances) ដោយប្រើ Free GPU (T4 GPU) លើ Google Colab។

**ជំហានចម្បង៖**
1. ត្រួតពិនិត្យ GPU (NVIDIA T4 / V100 / A100)
2. Clone Project & តម្លើង Libraries
3. Download & Extract Dataset OpenSLR 42
4. Preprocess Dataset & Resample Audio (22.05 kHz)
5. ស្រង់ Durations តាមបែប Acoustic Energy Alignment
6. Train FastSpeechLite (Stage 1: Acoustic Model)
7. Train HiFiGAN-tiny (Stage 2: Neural Vocoder)
8. សាកល្បងសំយោគសំឡេង (Inference & Audio Player)
9. Export ទៅកាន់ ONNX & Backup ចូល Google Drive

## ជំហានទី ១: ត្រួតពិនិត្យ GPU
*(ត្រូវប្រាកដថាបានជ្រើសរើស **Runtime -> Change runtime type -> T4 GPU**)*

In [ ]:
!nvidia-smi

## ជំហានទី ២: Clone Repository & តម្លើង Dependencies

In [ ]:
# Clone Repository ពី GitHub
!git clone https://github.com/therealphearunpo/KHM-TtS.git
%cd KHM-TtS

# តម្លើង Python Packages
!pip install -r requirements.txt

## ជំហានទី ៣: ទាញយក Dataset OpenSLR 42 (km_kh_male)
ទិន្នន័យមានទំហំប្រមាណ 853 MB មានសំឡេងខ្មែរចំនួន 2,906 utterances។

In [ ]:
# ទាញយក Dataset ផ្ទាល់ពី OpenSLR
!wget -q --show-progress https://www.openslr.org/resources/42/km_kh_male.zip

# ស្រង់ឯកសារចេញ
!unzip -q km_kh_male.zip
!echo "Dataset extracted successfully! Total files:"
!ls -1 km_kh_male/wavs/*.wav | wc -l

## ជំហានទី ៤: Preprocess Dataset (Resample 22.05 kHz & Mel Computation)
ដំណើរការសម្អាត Audio Trim Silent Edges, Peak Normalization (0.95), និងទាញយក 80-band Mel-spectrograms។

In [ ]:
!python data/prepare_dataset.py --data_dir km_kh_male --out_dir processed

## ជំហានទី ៥: ស្រង់ Acoustic Energy-Guided Durations
គណនា Log Mel Energy ដើម្បីតម្រឹម Token នីមួយៗទៅនឹងរយៈពេលសង្កត់សំឡេងពិតប្រាកដក្នុង Waveform។

In [ ]:
!python data/extract_durations.py --data_dir km_kh_male --proc_dir processed

## ជំហានទី ៦: Train FastSpeechLite (Acoustic Model - Stage 1)
បង្វឹកគំរូ Acoustic Model (~4M parameters) សម្រាប់បំប្លែង Khmer Graphemes ទៅកាន់ Mel-Spectrogram។

In [ ]:
!python train_acoustic.py \
    --data_dir km_kh_male \
    --epochs 100 \
    --batch_size 16 \
    --lr 0.001 \
    --save_interval 10

## ជំហានទី ៧: Train HiFiGAN-tiny (Neural Vocoder - Stage 2)
បង្វឹក Neural Vocoder (~1.5M parameters) ដើម្បីបំប្លែង Mel-Spectrogram ត្រឡប់មកជាសម្លេង 22,050 Hz Waveform ដ៏ពិរោះ។

In [ ]:
!python train_vocoder.py \
    --data_dir km_kh_male \
    --epochs 200 \
    --batch_size 16 \
    --lr 0.0002 \
    --save_interval 20

## ជំហានទី ៨: សាកល្បងសំយោគសំឡេង (Inference & Audio Test)

In [ ]:
from IPython.display import Audio, display

# សំយោគអត្ថបទសាកល្បង
!python infer.py \
    --text "សូមស្វាគមន៍មកកាន់ប្រព័ន្ធសំយោគសំឡេងខ្មែរឆ្លាតវៃ ឆ្នាំ ២០២៦។" \
    --acoustic_ckpt checkpoints/acoustic_best.pt \
    --vocoder_ckpt checkpoints/vocoder_best.pt \
    --output output_sample.wav

# ចាក់សំឡេងស្ដាប់ផ្ទាល់ក្នុង Colab
Audio("output_sample.wav")

## ជំហានទី ៩: Export ទៅកាន់ ONNX សម្រាប់ Web Studio

In [ ]:
!python export_onnx.py \
    --acoustic_ckpt checkpoints/acoustic_best.pt \
    --vocoder_ckpt checkpoints/vocoder_best.pt \
    --out_dir web/models

## ជំហានទី ១០: រក្សាទុក Checkpoints ចូលក្នុង Google Drive
ដើម្បីកុំឱ្យបាត់បង់ទិន្នន័យនៅពេល Colab ផ្ដាច់ session សូម Save ចូល Google Drive ផ្ទាល់ខ្លួន៖

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# បង្កើត Folder ក្នុង Drive និង Copy Checkpoints ទុក
!mkdir -p /content/drive/MyDrive/KHM_TTS_Checkpoints
!cp -r checkpoints/* /content/drive/MyDrive/KHM_TTS_Checkpoints/
!cp -r web/models/*.onnx /content/drive/MyDrive/KHM_TTS_Checkpoints/
!echo "All models and checkpoints saved to Google Drive successfully!"